# Colección y Descripción de Datos — Internet Fijo Accesos por Tecnología y Segmento

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, regexp_replace, mean, sum as Fsum

In [0]:
spark = SparkSession.builder.appName("MinEducacion_Proyecto").getOrCreate()

#RUTA ANGE /Volumes/workspace/proyecto/proyectvolume/dataframes/dataframes/Internet-Fijo-Accesos-por-Tecnología-y-Segmento.csv
#RUTA NATALIA /Volumes/catalogproyectosparkies/default/volumeproyectosparkies/Internet-Fijo-Accesos-por-Tecnología-y-Segmento.csv


file_path = "/Volumes/catalogproyectosparkies/default/volumeproyectosparkies/Internet-Fijo-Accesos-por-Tecnología-y-Segmento.csv"
df_internet = spark.read.csv(file_path, header=True, inferSchema=False, sep=",")

display(df_internet.limit(15))
# para verlos todos se qutia el limit
#display(df_internet)

In [0]:
print("--- ESQUEMA DE DATOS ---")
df_internet.printSchema()

print(f"\nTotal de registros (filas): {df_internet.count()}")
print(f"Total de atributos (columnas): {len(df_internet.columns)}")

## Comprensión del significado de atributos clave:

**AÑO / TRIMESTRE**: Período de reporte del dato (2016–2023, trimestres 1 a 4).

**PROVEEDOR**: Empresa prestadora del servicio de internet fijo.

**COD_DEPARTAMENTO / DEPARTAMENTO**: Código y nombre del departamento (15 = Boyacá).

**COD_MUNICIPIO / MUNICIPIO**: Código DANE y nombre del municipio donde se presta el servicio.

**SEGMENTO**: Clasificación del suscriptor: residencial (estratos 1–6), corporativo, sin estratificar o uso propio del operador.

**TECNOLOGIA**: Tecnología usada para la conexión (FTTH, CABLE, XDSL, SATELITAL, WIFI, entre otras).

**VELOCIDAD_BAJADA / VELOCIDAD_SUBIDA**: Velocidad contratada en Mbps para descarga y carga. Vienen con coma decimal en lugar de punto.

**No DE ACCESOS**: Número de suscriptores activos para esa combinación de proveedor, municipio, tecnología y segmento en ese período.

#Exploración de los Datos
Se presentan 8 elementos exploratorios para comprender la distribución
de la conectividad fija en los municipios de Boyacá entre 2016 y 2023.

In [0]:
df_exp = df_internet \
    .withColumn("VELOCIDAD_BAJADA_NUM", regexp_replace(col("VELOCIDAD_BAJADA"), ",", ".").cast("float")) \
    .withColumn("VELOCIDAD_SUBIDA_NUM", regexp_replace(col("VELOCIDAD_SUBIDA"), ",", ".").cast("float")) \
    .withColumn("NUM_ACCESOS", col("`No DE ACCESOS`").cast("integer"))

#1: Estadísticos descriptivos de velocidades y accesos
print("Elemento 1: Estadísticos de Velocidad de Bajada (Mbps)")
df_exp.select("VELOCIDAD_BAJADA_NUM").summary().show()

print("Elemento 2: Estadísticos de Velocidad de Subida (Mbps)")
df_exp.select("VELOCIDAD_SUBIDA_NUM").summary().show()

#3: Registros por año
print("Elemento 3: Cantidad de registros por Año")
df_exp.groupBy("AÑO").count().orderBy("AÑO").show()

#4: Total de accesos por año
print("Elemento 4: Total de accesos (suscriptores) por Año")
df_exp.groupBy("AÑO") \
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos")) \
    .orderBy("AÑO").show()

In [0]:
#5: Evolución de accesos por año
display(
    df_exp.groupBy("AÑO")
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos"))
    .orderBy("AÑO")
)

#6: Accesos por segmento
display(
    df_exp.groupBy("SEGMENTO")
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos"))
    .orderBy("total_accesos", ascending=False)
)

#7: Accesos por tecnología
display(
    df_exp.groupBy("TECNOLOGIA")
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos"))
    .orderBy("total_accesos", ascending=False)
)

#8: Top 10 municipios por accesos
display(
    df_exp.groupBy("MUNICIPIO")
    .agg(Fsum("NUM_ACCESOS").alias("total_accesos"))
    .orderBy("total_accesos", ascending=False)
    .limit(10)
)

#Reporte de Calidad de Datos

In [0]:
from pyspark.sql.functions import col, count, when, isnan

print("--- REPORTE DE VALORES FALTANTES ---")

expresiones_nulos = []
for columna, tipo in df_internet.dtypes:
    if tipo == 'string':
        condicion = col(columna).isNull() | (col(columna) == "") | (col(columna) == "NaN")
    else:
        condicion = col(columna).isNull() | isnan(columna)
    expresiones_nulos.append(count(when(condicion, columna)).alias(columna))

df_internet.select(expresiones_nulos).show(vertical=True)

In [0]:
df_exp.select(
    count(when(col("VELOCIDAD_BAJADA_NUM") == 0, 1)).alias("velocidad_bajada_cero"),
    count(when(col("VELOCIDAD_SUBIDA_NUM") == 0, 1)).alias("velocidad_subida_cero"),
    count(when(col("NUM_ACCESOS") == 0, 1)).alias("accesos_cero")
).show()

print("Registros por segmento a eliminar:")
df_internet.filter(
    col("SEGMENTO").isin("USO PROPIO INTERNO DEL OPERADOR", "SIN ESTRATIFICAR")
).groupBy("SEGMENTO").count().show(truncate=False)

## Técnicas propuestas para tratar valores problemáticos:

#### Eliminación de registros con accesos == 0:
Representan combinaciones de proveedor/municipio/tecnología sin suscriptores reales.
No aportan información al análisis de cobertura y son eliminados con un filtro.

#### Eliminación del segmento "USO PROPIO INTERNO DEL OPERADOR":
Solo 161 registros (0.1%) que corresponden a uso interno de los operadores,
no representan acceso real.

#### Eliminación de columnas redundantes:
COD_DEPARTAMENTO y DEPARTAMENTo se eliminan porque el dataset ya está
filtrado a Boyacá.

#### Velocidades en 0: 
Se dejan como nulos en lugar de cero para promedios en análisis futuros.

#Filtros, Limpieza y Transformación Inicial

In [0]:
print(f"Filas antes de filtros: {df_internet.count()}")

df_internet = df_internet \
    .withColumn("VELOCIDAD_BAJADA", regexp_replace(col("VELOCIDAD_BAJADA"), ",", ".").cast("float")) \
    .withColumn("VELOCIDAD_SUBIDA", regexp_replace(col("VELOCIDAD_SUBIDA"), ",", ".").cast("float")) \
    .withColumn("No DE ACCESOS", col("`No DE ACCESOS`").cast("integer"))

#1: Eliminar accesos == 0
df_internet = df_internet.filter(col("No DE ACCESOS") > 0)
print(f"Filas tras eliminar accesos == 0: {df_internet.count()}")

#2: Eliminar segmento
df_internet = df_internet.filter(col("SEGMENTO") != "USO PROPIO INTERNO DEL OPERADOR")
print(f"Filas tras eliminar uso propio operador: {df_internet.count()}")

In [0]:
#1: Eliminar columnas redundantes
df_internet = df_internet.drop("COD_DEPARTAMENTO", "DEPARTAMENTO")
print(f"Columnas tras eliminación: {len(df_internet.columns)}")

#2: 0 = null
df_internet = df_internet \
    .withColumn("VELOCIDAD_BAJADA", when(col("VELOCIDAD_BAJADA") == 0, None).otherwise(col("VELOCIDAD_BAJADA"))) \
    .withColumn("VELOCIDAD_SUBIDA", when(col("VELOCIDAD_SUBIDA") == 0, None).otherwise(col("VELOCIDAD_SUBIDA")))

# 3: Creacion d columna
df_internet = df_internet.withColumn(
    "TIPO_TECNOLOGIA",
    when(col("TECNOLOGIA").contains("FIBER") | col("TECNOLOGIA").contains("FTTX"), "FIBRA")
    .when(col("TECNOLOGIA").isin("CABLE", "HYBRID FIBER COAXIAL (HFC)"), "CABLE/HFC")
    .when(col("TECNOLOGIA") == "XDSL", "XDSL")
    .when(col("TECNOLOGIA") == "SATELITAL", "SATELITAL")
    .when(col("TECNOLOGIA").isin("WIFI", "WIMAX", "OTRAS TECNOLOGÍAS INALÁMBRICAS"), "INALÁMBRICO")
    .otherwise("OTRAS")
)

print("Accesos por TIPO_TECNOLOGIA:")
df_internet.groupBy("TIPO_TECNOLOGIA") \
    .agg(Fsum("No DE ACCESOS").alias("total_accesos")) \
    .orderBy("total_accesos", ascending=False).show()

#4: Creacion columna
df_internet = df_internet.withColumn(
    "ES_RESIDENCIAL",
    when(col("SEGMENTO").startswith("RESIDENCIAL"), 1).otherwise(0)
)

print("Distribución ES_RESIDENCIAL:")
df_internet.groupBy("ES_RESIDENCIAL").agg(
    count("*").alias("registros"),
    Fsum("No DE ACCESOS").alias("total_accesos")
).show()

In [0]:
print("--- REPORTE DE NULOS TRAS TRANSFORMACIONES ---")

expresiones_finales = []
for columna, tipo in df_internet.dtypes:
    if tipo == 'string':
        condicion = col(columna).isNull() | (col(columna) == "") | (col(columna) == "NaN")
    else:
        condicion = col(columna).isNull()
    expresiones_finales.append(count(when(condicion, columna)).alias(columna))

df_internet.select(expresiones_finales).show(vertical=True)

print("--- MUESTRA DEL DATASET TRANSFORMADO Y LIMPIO ---")
df_internet.select(
    "AÑO", "MUNICIPIO", "PROVEEDOR", "SEGMENTO",
    "TIPO_TECNOLOGIA", "VELOCIDAD_BAJADA", "VELOCIDAD_SUBIDA",
    "No DE ACCESOS", "ES_RESIDENCIAL"
).show(5, truncate=False)

print(f"\nFilas finales: {df_internet.count()}")
print(f"Columnas finales: {len(df_internet.columns)}")

In [0]:
display(df_internet)

In [0]:
df_internet.groupBy("MUNICIPIO").count().orderBy("count", ascending=False).show()